# 实验报告03：正则表达式与网络结构化数据

学号：2024408029　姓名：何启恒
按顺序运行各代码单元。

In [1]:
import hashlib
import os
import platform
import uuid
import sys

# 修改为自己的学号
STUDENT_ID = 2024408029

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")
print(f"[Python]    {sys.version.split()[0]}")


[Conda 环境] None
[学号]       2024408029
[机器指纹] 8c49fe190c554f6a
[RUN_ID]   2024408029-8c49fe190c554f6a
[Python]    3.13.18


## 实验1：正则表达式的基础匹配

导入 `re`，在给定文本上练习字符类、空白符、单词边界、分支与量词。

In [2]:
import re


### 1. 数字、空白符与单词边界

- `\d+`：匹配一个或多个数字；
- `\s+`：匹配一个或多个空白字符（空格、制表符、换行等）；
- `\bcat\b`：用单词边界 `\b` 精确匹配独立单词 `cat`，不会匹配到 `cats` 内部。

In [3]:
text1 = "cat on mat; later: cats and dogs"

digits = re.findall(r"\d+", text1)
spaces = re.findall(r"\s+", text1)
cats = re.findall(r"\bcat\b", text1)

print("文本：", text1)
print(r"\d+ 数字匹配：", digits)
print(r"\s+ 空白匹配：", spaces)
print(r"cat 独立单词 cat：", cats)


文本： cat on mat; later: cats and dogs
\d+ 数字匹配： []
\s+ 空白匹配： [' ', ' ', ' ', ' ', ' ', ' ']
cat 独立单词 cat： ['cat']


### 2. 分支 `cat|dog` 与分支顺序的影响

`cat|dog` 表示匹配 `cat` 或 `dog`。`re.findall` 从左到右扫描，分支按书写顺序尝试；对于同一位置，先写在前的分支优先匹配。

对比 `a|aa|aaa` 与 `aaa|aa|a`：前者在每个位置先尝试最短的 `a`，匹配后继续向右；后者先尝试最长的 `aaa`，因此同一字符串上的切分结果不同。

In [4]:
text2 = "cat dog bird CAT"
print("cat|dog 匹配：", re.findall(r"cat|dog", text2))

print("a|aa|aaa 在 'aaaa' 上：", re.findall(r"a|aa|aaa", "aaaa"))
print("aaa|aa|a 在 'aaaa' 上：", re.findall(r"aaa|aa|a", "aaaa"))


cat|dog 匹配： ['cat', 'dog']
a|aa|aaa 在 'aaaa' 上： ['a', 'a', 'a', 'a']
aaa|aa|a 在 'aaaa' 上： ['aaa', 'a']


### 3. 量词 `* + ? {n} {m,n}`

- `ca*t`：`a` 出现 0 次或多次（`*`）；
- `ca+t`：`a` 出现 1 次或多次（`+`）；
- `ca?t`：`a` 出现 0 次或 1 次（`?`）；
- `ca{2}t`：`a` 恰好出现 2 次；
- `ca{1,2}t`：`a` 出现 1 到 2 次。

In [5]:
text3 = "ct cat caat caaat"

patterns = {
    "ca*t": r"ca*t",
    "ca+t": r"ca+t",
    "ca?t": r"ca?t",
    "ca{2}t": r"ca{2}t",
    "ca{1,2}t": r"ca{1,2}t",
}

for name, pattern in patterns.items():
    print(f"{name:10s} -> {re.findall(pattern, text3)}")


ca*t       -> ['ct', 'cat', 'caat', 'caaat']
ca+t       -> ['cat', 'caat', 'caaat']
ca?t       -> ['ct', 'cat']
ca{2}t     -> ['caat']
ca{1,2}t   -> ['cat', 'caat']


## 实验2：分组、回溯引用与替换

使用捕获组提取子串，用后向引用匹配重复内容，并用 `re.sub()` 完成日期与电话号码的格式化。

### 1. 捕获组提取邮箱用户名与域名

邮箱格式为 `用户名@域名`。正则 `([\w.]+)@([\w.]+)` 中：
第 1 个捕获组 `([\w.]+)` 提取用户名，第 2 个捕获组 `([\w.]+)` 提取域名。

In [6]:
email_text = "联系：alice@example.com；备选：bob@test.org"

emails = re.findall(r"([\w.]+)@([\w.]+)", email_text)
for username, domain in emails:
    print(f"邮箱：{username}@{domain}  用户名：{username}  域名：{domain}")


邮箱：alice@example.com  用户名：alice  域名：example.com
邮箱：bob@test.org  用户名：bob  域名：test.org


### 2. 后向引用找出连续重复的单词

`(\b\w+\b)\s+\1` 中，`(\b\w+\b)` 捕获一个单词，`\1` 后向引用第 1 个捕获组，用于匹配与前面完全相同、中间只隔空白的重复单词。

In [7]:
repeat_text = "cat cat, dog cat, bird bird"

repeated = re.findall(r"(\b\w+\b)\s+\1", repeat_text)
print("连续重复的单词：", repeated)


连续重复的单词： ['cat', 'bird']


### 3. 日期格式：年-月-日 → 月-日-年

用捕获组分别提取年、月、日，在 `re.sub()` 的替换串中通过 `\2-\3-\1` 重新排列，保留原来的换行与行数。

In [8]:
date_text = "2024-05-01\n2024/05/02\n2024.05.03"
print("转换前：")
print(date_text)

# 同时兼容 - / . 三种分隔符；第1组年、第2组月、第3组日
converted = re.sub(
    r"(\d{4})[-/.](\d{1,2})[-/.](\d{1,2})",
    r"\2-\3-\1",
    date_text,
)
print("\n转换后：")
print(converted)


转换前：
2024-05-01
2024/05/02
2024.05.03

转换后：
05-01-2024
05-02-2024
05-03-2024


### 4. 电话号码统一为 `(123) 456-7890`

原号码分隔符可能是 `-` 或 `.`。捕获组分别提取前 3 位、中间 3 位、后 4 位，替换为 `(前缀) 中段-后段` 的统一格式。

In [9]:
phone_text = "联系我：123-456-7890 或 987.654.3210"
print("转换前：", phone_text)

normalized = re.sub(
    r"(\d{3})[-.](\d{3})[-.](\d{4})",
    r"(\1) \2-\3",
    phone_text,
)
print("转换后：", normalized)


转换前： 联系我：123-456-7890 或 987.654.3210
转换后： 联系我：(123) 456-7890 或 (987) 654-3210


## 实验3：使用 BeautifulSoup 解析 HTML

使用给定 HTML 字符串，不依赖网页下载；用 `BeautifulSoup(html_doc, "html.parser")` 构建解析树。

In [10]:
from bs4 import BeautifulSoup

html_doc = """
<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲：正则表达式</p>
<p class="course">第七讲：结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>
"""

soup = BeautifulSoup(html_doc, "html.parser")


### 1. 标题与 h1 信息

`soup.title` 取得 `<title>` 节点，`.text`（或 `.get_text()`）取其文本；`soup.h1` 取得 `<h1>` 节点，`.name` 为标签名，`.text` 为标签内文本。

In [11]:
title_tag = soup.title
h1_tag = soup.h1

print("页面标题文本：", title_tag.text)
print("h1 标签名：", h1_tag.name)
print("h1 文本：", h1_tag.text)


页面标题文本： 实验课程
h1 标签名： h1
h1 文本： 数据分析


### 2. `find_all()` 提取课程段落与链接

- `soup.find_all('p', class_='course')`：找出所有 `class='course'` 的 `<p>` 段落；
- `soup.find_all('a')`：找出所有链接，逐项输出链接文字与 `href` 属性。

In [12]:
course_paragraphs = soup.find_all("p", class_="course")
print('class="course" 的段落：')
for p in course_paragraphs:
    print(" -", p.get_text())

links = soup.find_all("a")
print("\n所有链接：")
for a in links:
    print(f" - 链接文字：{a.get_text()}，href：{a.get('href')}")


class="course" 的段落：
 - 第六讲：正则表达式
 - 第七讲：结构化数据

所有链接：
 - 链接文字：讲义六，href：https://example.org/lecture06
 - 链接文字：讲义七，href：https://example.org/lecture07


### 3. `get_text()` 提取可见文本并比较

分别对整个文档、单个段落、单个链接调用 `get_text()`，比较三者返回文本的范围：整个文档包含所有可见文本，单个段落只含该段落文本，单个链接只含链接文字。

In [13]:
doc_text = soup.get_text()
first_p = soup.find("p", class_="course")
first_a = soup.find("a")

print("整个文档 get_text()：")
print(repr(doc_text))
print("\n单个段落 get_text()：", repr(first_p.get_text()))
print("单个链接 get_text()：", repr(first_a.get_text()))


整个文档 get_text()：
'\n实验课程\n数据分析\n第六讲：正则表达式\n第七讲：结构化数据\n讲义六\n讲义七\n\n'

单个段落 get_text()： '第六讲：正则表达式'
单个链接 get_text()： '讲义六'
